In [1]:
import os
import pandas as pd
import numpy as np

In [ ]:
base_fold = '/home/h/Harini.Maruthi/Data'
fold_bam = base_fold + '/Bam_files/BigWig/src/clean'
os.chdir(fold_bam)
files_bam = [file for file in os.listdir() if '.csv' in file]

types = np.unique([file.split('_')[0] for file in files_bam])
samples = np.unique([file.split('_')[1]+'_'+file.split('_')[2] for file in files_bam])

file_dict = {}
for t in types:
    file_dict[t] = {}
    files_t = [file for file in files_bam if t in file]
    for sample in samples:
        file_dict[t][sample] = [file for file in files_t if sample in file]

In [2]:
def average_replicate_matrices(data_dict, matrix_dir, load_func=pd.read_csv):
    """
    Load and average coverage matrices (forward & reverse) for each experiment.
    
    Args:
        data_dict: nested dict {experiment: {replicate: [reverse_path, forward_path]}}
        matrix_dir: path to directory containing coverage matrix files
        load_func: function to load matrix file (default: pd.read_csv)
    
    Returns:
        dict_mat: {experiment: {'forward': DataFrame, 'reverse': DataFrame}}
    """
    dict_mat = {}

    for experiment, replicates in data_dict.items():
        forward_matrices = []
        reverse_matrices = []

        for rep_name, files in replicates.items():
            # Identify which file is forward/reverse
            forward_file = next(f for f in files if "forward" in f)
            reverse_file = next(f for f in files if "reverse" in f)

            # Load the matrices
            forward_path = os.path.join(matrix_dir, forward_file)
            reverse_path = os.path.join(matrix_dir, reverse_file)

            fwd_df = load_func(forward_path, index_col=0)
            rev_df = load_func(reverse_path, index_col=0)

            forward_matrices.append(fwd_df)
            reverse_matrices.append(rev_df)

        # Compute mean across replicates (aligned by index/columns)
        forward_avg = sum(forward_matrices) / len(forward_matrices)
        forward_avg.to_csv(f'avgforwardmatrix{experiment}.csv')
        reverse_avg = sum(reverse_matrices) / len(reverse_matrices)
        reverse_avg.to_csv(f'avgreversematrix{experiment}.csv')

        dict_mat[experiment] = {
            "forward": forward_avg,
            "reverse": reverse_avg
        }

    return dict_mat

In [3]:
def compute_cumulative_dict(dict_mat, output_dir):
    """
    Compute and return cumulative matrices per experiment.
    Also saves forward and reverse CSVs to disk.

    Args:
        dict_mat: dict of {experiment: {'forward': DataFrame, 'reverse': DataFrame}}
        output_dir: directory to save CSVs

    Returns:
        cumulative_dict: same structure as dict_mat but with cumulative values
    """
    os.makedirs(output_dir, exist_ok=True)
    cumulative_dict = {}

    for experiment, matrices in dict_mat.items():
        forward = matrices['forward'].iloc[:, :-1]  # remove dummy column if present
        reverse = matrices['reverse'].iloc[:, :-1]

        fwd_cum = pd.DataFrame(index=forward.index, columns=forward.columns)
        rev_cum = pd.DataFrame(index=reverse.index, columns=reverse.columns)

        for idx in forward.index:
            # Forward: right to left
            fwd_row = forward.loc[idx].cumsum()
            #fwd_row = forward.loc[idx][::-1].cumsum()[::-1]
            total_fwd = fwd_row.iloc[-1]
            fwd_row /= total_fwd if total_fwd != 0 else 1
            fwd_cum.loc[idx] = fwd_row

            # Reverse: left to right
            #rev_row = reverse.loc[idx].cumsum()
            rev_row = reverse.loc[idx][::-1].cumsum()[::-1]
            total_rev = rev_row.iloc[0]
            rev_row /= total_rev if total_rev != 0 else 1
            rev_cum.loc[idx] = rev_row

        # Save CSVs
        fwd_path = os.path.join(output_dir, f"{experiment}_forward_cumulative.csv")
        rev_path = os.path.join(output_dir, f"{experiment}_reverse_cumulative.csv")
        fwd_cum.to_csv(fwd_path)
        rev_cum.to_csv(rev_path)
        print(f"[✓] Saved: {fwd_path}")
        print(f"[✓] Saved: {rev_path}")

        # Store in new dict
        cumulative_dict[experiment] = {
            'forward': fwd_cum,
            'reverse': rev_cum
        }

    return cumulative_dict


In [ ]:
output_dir = "avgcumulative"
cumulative_dict = compute_cumulative_dict(dict_mat, output_dir)

In [4]:
def compute_crossings(dict_mat):
    """
    Compute horizontal and vertical crossings for each region and experiment.
    
    For each row, crossings are defined as:
    - Horizontal: column index where forward and reverse are equal, or mean(j, j+1) if a crossing occurs between.
    - Vertical: value of the forward/reverse matrix at that crossing.

    Returns:
        DataFrame with columns: horizontal_<experiment>, vertical_<experiment>
    """
    all_crossings = {}

    for experiment, matrices in dict_mat.items():
        fwd_df = matrices['forward']
        rev_df = matrices['reverse']

        fwd = fwd_df.to_numpy()
        rev = rev_df.to_numpy()
        index = fwd_df.index
        cols = np.array(fwd_df.columns, dtype=float)

        horizontal_crossings = []
        vertical_crossings = []

        for i in range(fwd.shape[0]):
            crossing_points = []
            crossing_values = []

            for j in range(fwd.shape[1] - 1):
                # Exact crossing
                if fwd[i, j] == rev[i, j]:
                    crossing_points.append(cols[j])
                    crossing_values.append(fwd[i, j])

                # Crossover between columns
                elif (fwd[i, j] < rev[i, j] and fwd[i, j + 1] > rev[i, j + 1]) or \
                     (fwd[i, j] > rev[i, j] and fwd[i, j + 1] < rev[i, j + 1]):
                    # Use average position between columns
                    crossing_pos = (cols[j] + cols[j + 1]) / 2
                    crossing_val = np.mean([fwd[i, j], fwd[i, j + 1]])
                    crossing_points.append(crossing_pos)
                    crossing_values.append(crossing_val)

            # Mean of all crossings in the row
            horizontal_crossings.append(np.mean(crossing_points) if crossing_points else np.nan)
            vertical_crossings.append(np.mean(crossing_values) if crossing_values else np.nan)

        # Store in output dict
        all_crossings[f'horizontal_{experiment}'] = horizontal_crossings
        all_crossings[f'vertical_{experiment}'] = vertical_crossings

    return pd.DataFrame(all_crossings, index=index)

In [5]:
def compute_spearman_per_region(crossings_df, experiment_order):
    """
    Compute Spearman correlation per region for horizontal and vertical crossings.
    
    Returns:
        DataFrame with columns: region, type, spearman_corr, p_value
    """
    timepoints = list(range(len(experiment_order)))
    results = []

    for crossing_type in ['horizontal', 'vertical']:
        cols = [f"{crossing_type}_{exp}" for exp in experiment_order]
        sub_df = crossings_df[cols]

        for idx in sub_df.index:
            values = sub_df.loc[idx].values.astype(float)
            mask = ~np.isnan(values)
            if mask.sum() < 2:
                corr, pval = np.nan, np.nan
            else:
                corr, pval = spearmanr(np.array(timepoints)[mask], values[mask])
            results.append({
                'region': idx,
                'type': crossing_type,
                'spearman_corr': corr,
                'p_value': pval
            })

    return pd.DataFrame(results)


import matplotlib.pyplot as plt
import seaborn as sns

def plot_significant_spearman_histogram(spearman_df):
    """
    Plot histogram + KDE of Spearman ρ values (p < 0.05),
    separately for horizontal and vertical.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    for ax, ctype in zip(axes, ['horizontal', 'vertical']):
        subset = spearman_df[
            (spearman_df['type'] == ctype) &
            (spearman_df['p_value'] < 0.05) &
            (~spearman_df['spearman_corr'].isna())
        ]

        if not subset.empty:
            sns.histplot(
                subset['spearman_corr'],
                bins=20,
                kde=True,
                stat="count",
                edgecolor='black',
                alpha=0.6,
                ax=ax
            )
        else:
            ax.text(0.5, 0.5, "No significant\ncorrelations", ha='center', va='center', fontsize=12)
            ax.set_xlim(-1, 1)

        ax.set_title(f"{ctype.capitalize()} crossings (p < 0.05)")
        ax.set_xlabel("Spearman ρ")
        ax.set_ylabel("Count")
        ax.axvline(0, color='black', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.show()


In [6]:
experiment_order = ['matrixRPA-WT-164', 'matrixRPA-WT-176', 'matrixRPA-WT-190', 'matrixRPA-WT-212', 'matrixRPA-WT-236']

spearman_df_rpa = compute_spearman_per_region(crossings_df_rpa, experiment_order)

plot_significant_spearman_histogram(spearman_df_rpa)


NameError: name 'crossings_df_rpa' is not defined

In [7]:
experiment_order_dmc1 = ['matrixDMC1-WT-164', 'matrixDMC1-WT-176', 'matrixDMC1-WT-190', 'matrixDMC1-WT-212', 'matrixDMC1-WT-236']
experiment_order = ['matrixRPA-WT-164', 'matrixRPA-WT-176', 'matrixRPA-WT-190', 'matrixRPA-WT-212', 'matrixRPA-WT-236']

In [8]:
def plot_spearman_histograms(spearman_df):
    """
    Plot histograms and KDEs of Spearman ρ values for:
    - All correlations
    - Only those with p < 0.05
    Separately for horizontal and vertical.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

    for i, ctype in enumerate(['horizontal', 'vertical']):
        # All correlations
        all_subset = spearman_df[
            (spearman_df['type'] == ctype) &
            (~spearman_df['spearman_corr'].isna())
        ]
        sns.histplot(
            all_subset['spearman_corr'],
            bins=20,
            kde=True,
            stat="count",
            edgecolor='black',
            alpha=0.6,
            ax=axes[0, i]
        )
        axes[0, i].set_title(f"All {ctype} Spearman ρ")
        axes[0, i].axvline(0, color='black', linestyle='--')

        # Significant only (p < 0.05)
        sig_subset = all_subset[all_subset['p_value'] < 0.05]
        if not sig_subset.empty:
            sns.histplot(
                sig_subset['spearman_corr'],
                bins=20,
                kde=True,
                stat="count",
                edgecolor='black',
                alpha=0.6,
                ax=axes[1, i]
            )
        else:
            axes[1, i].text(0.5, 0.5, "No significant\ncorrelations", ha='center', va='center', fontsize=12)
            axes[1, i].set_xlim(-1, 1)

        axes[1, i].set_title(f"Significant {ctype} Spearman ρ (p < 0.05)")
        axes[1, i].axvline(0, color='black', linestyle='--')

        # Set axis labels
        for row in [0, 1]:
            axes[row, i].set_xlabel("Spearman ρ")
            axes[row, i].set_ylabel("Count")

    plt.tight_layout()
    plt.show()


In [9]:
def plot_crossing_trends_scatter(crossings_df, experiment_order):
    """
    Plot mean ± SEM, median, and individual data points (scatter) for
    horizontal and vertical crossing values over time.

    Args:
        crossings_df: DataFrame with columns like horizontal_<exp>, vertical_<exp>
        experiment_order: list of experiment names in temporal order
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

    for ax, crossing_type in zip(axes, ['horizontal', 'vertical']):
        means, medians, sems = [], [], []

        for i, exp in enumerate(experiment_order):
            col = f"{crossing_type}_{exp}"
            values = crossings_df[col].astype(float)
            means.append(values.mean(skipna=True))
            medians.append(values.median(skipna=True))
            sems.append(values.sem(skipna=True))

            # Scatter: add jittered x-values for visual separation
            y_vals = values.dropna()
            jittered_x = np.random.normal(loc=i, scale=0.1, size=len(y_vals))
            ax.scatter(jittered_x, y_vals, alpha=0.4, color='gray', s=15, label='Region values' if i == 0 else None)

        x = np.arange(len(experiment_order))
        means = np.array(means)
        medians = np.array(medians)
        sems = np.array(sems)

        ax.plot(x, means, color ='darkblue', label='Mean', marker='o', linestyle='-', linewidth=2)
        #ax.fill_between(x, means - sems, means + sems, alpha=0.3, label='± SEM')
        ax.plot(x, medians, color = 'darkred', label='Median', marker='s', linestyle='--', linewidth=2)

        ax.set_xticks(x)
        ax.set_xticklabels(experiment_order, rotation=45)
        ax.set_title(f"{crossing_type.capitalize()} Crossings Over Time")
        ax.set_ylabel("Crossing Value")
        ax.set_xlabel("Timecourse")
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [10]:
def plot_overlayed_coverage_with_background_subtraction_zeroed(cumulative_dict, experiment_order=None):
    if experiment_order is None:
        experiment_order = list(cumulative_dict.keys())

    plt.figure(figsize=(10, 6))
    cmap = cm.get_cmap('viridis', len(experiment_order))
    color_cycle = [cmap(i) for i in range(len(experiment_order))]

    after_subtraction = []
    bgsub_dict = {}

    for i, experiment in enumerate(experiment_order):
        # Drops the last column if it’s metadata, as in your code
        forward_df = cumulative_dict[experiment]['forward'].iloc[:, :-1]
        reverse_df = cumulative_dict[experiment]['reverse'].iloc[:, :-1]

        
        
        reverse_bg = reverse_df.iloc[:, 300:399].mean(axis=1, skipna=True)
        print(reverse_bg)
        print(reverse_bg.mean())
        reverse_df_bgsub = reverse_df.sub(reverse_bg, axis=0) #.clip(lower=0)
        reverse_df_bgsub = reverse_df_bgsub #.clip(lower=0)

        reverse_df_bgsub.iloc[:, 300:399] = 0.0

        after_subtraction.append(reverse_df_bgsub)

        check_rev = reverse_df_bgsub.iloc[:, 300:399].mean(axis=1)
        #print(check_rev)  # mean ≈ 0, small numerical noise
        #print(reverse_df_bgsub.iloc[:, -200:-150].mean(axis=0).mean()) #mean after averaging over all hotspots, still 0 is expected
        
        
        forward_bg = forward_df.iloc[:, 0:100].mean(axis=1, skipna=True)
        #print(forward_bg)
        forward_df_bgsub = forward_df.sub(forward_bg, axis=0) #.clip(lower=0)
        forward_df_bgsub = forward_df_bgsub #.clip(lower=0)
        forward_df_bgsub.iloc[:, 0:100] = 0.0

        bgsub_dict[experiment] = {
            "forward": forward_df_bgsub,
            "reverse": reverse_df_bgsub
        }

        reverse_mean = reverse_df_bgsub.mean(axis=0) #.rolling(window=10, center=True).mean()

        #print(reverse_mean.iloc[-200:-150].mean())

        forward_mean = forward_df_bgsub.mean(axis=0) #.rolling(window=10, center=True).mean()   #.iloc[:1, :]

        # Convert indices (positions) to numeric arrays
        x_rev = _to_numeric_index(reverse_mean.index)
        x_fwd = _to_numeric_index(forward_mean.index)

        color = color_cycle[i]

        # Plot with corrected labels: reverse=solid, forward=dashed
        plt.plot(x_rev, reverse_mean, label=f"{experiment} (forward)", linestyle='-',  color=color)
        plt.plot(x_fwd, forward_mean, label=f"{experiment} (reverse)", linestyle='--', color=color)

    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()
    
    return bgsub_dict

In [ ]:
bgsub_rpa = plot_overlayed_coverage_with_background_subtraction_zeroed(dict_mat_rpa, experiment_order)
bgsub_dmc1 = plot_overlayed_coverage_with_background_subtraction_zeroed(dict_mat_dmc1, experiment_order_dmc1)

In [ ]:
cumulative_rpa_bgsub = compute_cumulative_dict(bgsub_rpa, 'cumulative_csvs_bgsub')
cumulative_dmc1_bgsub = compute_cumulative_dict(bgsub_dmc1, 'cumulative_csvs_bgsub')

In [ ]:
plot_overlayed_cumulative(cumulative_rpa_bgsub, experiment_order)
plot_overlayed_cumulative(cumulative_dmc1_bgsub, experiment_order_dmc1)

In [ ]:
crossings_df_bgsub_rpa = compute_crossings(cumulative_rpa_bgsub)
crossings_df_bgsub_dmc1 = compute_crossings(cumulative_dmc1_bgsub)

In [ ]:
# Compute Spearman correlations
spearman_df_bgsub_rpa = compute_spearman_per_region(crossings_df_bgsub_rpa, experiment_order)

# Plot histogram of significant correlations
plot_significant_spearman_histogram(spearman_df_bgsub_rpa)

In [ ]:
# Compute Spearman correlations
spearman_df_bgsub_dmc1 = compute_spearman_per_region(crossings_df_bgsub_dmc1, experiment_order_dmc1)

# Plot histogram of significant correlations
plot_significant_spearman_histogram(spearman_df_bgsub_dmc1)

In [ ]:
plot_crossing_trends_scatter(crossings_df_bgsub_rpa, experiment_order)
plot_crossing_trends_scatter(crossings_df_bgsub_dmc1, experiment_order_dmc1)

In [ ]:
def get_percentile_position_dfs(cumulative_dict, percentile_low=0.1, percentile_high=0.9):
    """
    Compute x-positions where cumulative signal crosses low/high percentiles.
    Returns two wide-format DataFrames (forward & reverse) with one row per region,
    and two columns per experiment: <experiment>_10th and <experiment>_90th.

    Args:
        cumulative_dict: dict of {experiment: {'forward': df, 'reverse': df}}
        percentile_low: e.g. 0.1
        percentile_high: e.g. 0.9

    Returns:
        forward_df_wide, reverse_df_wide: wide-format DataFrames
    """
    forward_10th = {}
    forward_90th = {}
    reverse_10th = {}
    reverse_90th = {}

    for experiment, strands in cumulative_dict.items():
        fwd_df = strands['forward']
        rev_df = strands['reverse']
        x_vals = fwd_df.columns.astype(float)

        f10 = []
        f90 = []
        r10 = []
        r90 = []
        idx = []

        for region in fwd_df.index:
            idx.append(region)

            try:
                # Forward (right to left)
                fwd = fwd_df.loc[region].astype(float).values
                fwd_x = x_vals
                #fwd_x = x_vals[::-1]
                fwd_y = fwd
                x10_fwd = np.interp(percentile_low, fwd_y, fwd_x)
                x90_fwd = np.interp(percentile_high, fwd_y, fwd_x)
            except:
                x10_fwd = x90_fwd = np.nan
            f10.append(x10_fwd)
            f90.append(x90_fwd)

            try:
                # Reverse (left to right)
                rev = rev_df.loc[region].astype(float).values[::-1]
                rev_x = x_vals[::-1]
                x10_rev = np.interp(percentile_low, rev, rev_x)
                x90_rev = np.interp(percentile_high, rev, rev_x)
            except:
                x10_rev = x90_rev = np.nan
            r10.append(x10_rev)
            r90.append(x90_rev)

        forward_10th[f"{experiment}_10th"] = pd.Series(f10, index=idx)
        forward_90th[f"{experiment}_90th"] = pd.Series(f90, index=idx)
        reverse_10th[f"{experiment}_10th"] = pd.Series(r10, index=idx)
        reverse_90th[f"{experiment}_90th"] = pd.Series(r90, index=idx)

    # Combine columns
    forward_df = pd.concat([pd.DataFrame(forward_10th), pd.DataFrame(forward_90th)], axis=1)
    reverse_df = pd.concat([pd.DataFrame(reverse_10th), pd.DataFrame(reverse_90th)], axis=1)

    # Optional: sort columns
    forward_df = forward_df.reindex(sorted(forward_df.columns), axis=1)
    reverse_df = reverse_df.reindex(sorted(reverse_df.columns), axis=1)

    return forward_df, reverse_df

In [ ]:
def plot_percentile_trends_scatter(percentile_df, experiment_order):
    """
    Plot mean ± SEM, median, and individual data points (scatter) for
    10th and 90th percentile positions over time.

    Args:
        percentile_df: DataFrame with columns like <exp>_10th and <exp>_90th
        experiment_order: list of experiment names in temporal order (no suffix)
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
    percentile_types = ['10th', '90th']
    colors = {'10th': 'royalblue', '90th': 'crimson'}

    for ax, perc_type in zip(axes, percentile_types):
        means, medians, sems = [], [], []

        for i, exp in enumerate(experiment_order):
            col = f"{exp}_{perc_type}"
            values = percentile_df[col].astype(float)
            means.append(values.mean(skipna=True))
            medians.append(values.median(skipna=True))
            sems.append(values.sem(skipna=True))

            # Scatter: region-wise values with jitter
            y_vals = values.dropna()
            jittered_x = np.random.normal(loc=i, scale=0.1, size=len(y_vals))
            ax.scatter(jittered_x, y_vals, alpha=0.4, color='gray', s=15, label='Region values' if i == 0 else None)

        x = np.arange(len(experiment_order))
        means = np.array(means)
        medians = np.array(medians)
        sems = np.array(sems)

        ax.plot(x, means, color=colors[perc_type], label='Mean', marker='o', linestyle='-', linewidth=2)
        # Optional SEM ribbon
        # ax.fill_between(x, means - sems, means + sems, alpha=0.2, color=colors[perc_type], label='± SEM')
        ax.plot(x, medians, color='black', label='Median', marker='s', linestyle='--', linewidth=2)

        ax.set_xticks(x)
        ax.set_xticklabels(experiment_order, rotation=45)
        ax.set_title(f"{perc_type} Percentile Positions Over Time")
        ax.set_ylabel("Genomic Position")
        ax.set_xlabel("Timecourse")
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
fwd_percentile_df_rpa_bgsub, rev_percentile_df_rpa_bgsub = get_percentile_position_dfs(cumulative_rpa_bgsub)

In [ ]:
fwd_percentile_df_dmc1_bgsub, rev_percentile_df_dmc1_bgsub = get_percentile_position_dfs(cumulative_dmc1_bgsub)

In [ ]:
plot_percentile_trends_scatter(rev_percentile_df_rpa_bgsub, experiment_order)
plot_percentile_trends_scatter(fwd_percentile_df_rpa_bgsub, experiment_order)

In [ ]:
plot_percentile_trends_scatter(rev_percentile_df_dmc1_bgsub, experiment_order_dmc1)
plot_percentile_trends_scatter(fwd_percentile_df_dmc1_bgsub, experiment_order_dmc1)